# Colab 23 — Retrieval (rank-aware, oracle-set) + embedding-space analysis

Builds directly on **colab22** (same two frozen encoders, same 2x3 {AA, SS}-encoder x
{AA, SS, 3Di}-feed matrix, same AUROC accuracy chart). Two things change:

### 1. Retrieval metric overhaul (replaces the hits@10 / frac@10 split)

colab22 used **hits@10** for AA (one designated partner) and **frac@10** for SS/3Di (set-based)
— two different metrics on one axis, and `frac@k = retrieved/min(k,|T|)` silently flips between
precision-like (crowded) and recall-like (sparse). colab23 uses **one metric family, uniformly**:

- **Relevance = the exact-Levenshtein oracle neighbour set** per query (every pool protein with
  exact `normLev >= 0.70`). This is the metric the encoder actually approximates, so a perfect
  Lev-approximator scores 1.0 — the only ceiling is the encoder's own error, not task crowding.
- **No AA special-casing.** The oracle scan defines `|T|` per query for *every* feed; AA simply
  falls out at `|T|~1` (only ~5 high-AA pairs exist in CATH), so its recall collapses to hits
  automatically.
- Metrics per (encoder, feed): **MAP@10** (headline — rank-aware, supervisor's ask),
  **R-precision** (give exactly `|T|` slots -> "of N partners, found M"), **recall@10 / @50**,
  **precision@10**, **hit-rate@10** (found *any* — the lenient floor), and `median |T|` (crowding).

### 2. Embedding-space analysis (revives + extends the colab16 UMAP)

The colab16 pool-UMAP with the 5 high-AA pairs drawn as red segments, brought back with a broader
read of the space: *which UMAP axis encodes length vs composition*, *what the other sequences
sitting near a high-sim pair actually are*, and a direct test of the horizontal-vs-vertical
question — does moving along one axis change length while the other changes character content?

Colours: **AA-encoder = blue**, **SS-encoder = green**.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_eval.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn scipy umap-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLevenshtein
from rapidfuzz.process import cdist as rf_cdist

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Presentation palette (consistent across both charts)
COL_AA_ENC = '#3b7dd8'   # blue  = AA-encoder
COL_SS_ENC = '#36a85a'   # green = SS-encoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants (encoder recipe identical to colab19 / colab17b)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
NUM_EPOCHS = 30
BATCH_SIZE = 128
K = 16

BAND_LOW_AA = 0.30
BAND_LOW_SS = 0.56
BAND_HIGH   = 0.70          # high-sim cutoff (positives for AUROC; oracle true-set bar)
BIN_NAMES = ['far', 'mid', 'high']

K_RETRIEVAL = 10            # @k for both hits@10 and frac@10
SEED_TRAIN_AA = 42
SEED_TRAIN_SS = 43

print(f'AA encoder BAND_LOW={BAND_LOW_AA} | SS encoder BAND_LOW={BAND_LOW_SS} '
      f'| high-sim cutoff={BAND_HIGH} | retrieval @k={K_RETRIEVAL}')

## 3. Helpers — Levenshtein, encode, perturb (RNG threaded for determinism)

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); choices = [c for c in abc if c != s[i]]; s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def random_seq(alphabet, rng, min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(alphabet), size=length))

def bin_idx_for(x, band_low):
    if x < band_low: return 0
    if x < BAND_HIGH: return 1
    return 2

def make_full_range_pairs(alphabet, n_pairs, rng):
    """t ~ U[0,1] -> roughly uniform coverage of normLev (far/mid/high all present)."""
    pairs = []; attempts = 0; max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed); t = float(rng.uniform(0.0, 1.0)); k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

## 4. Architecture + training (3-bin CE head, identical to colab19)

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32,
                 hidden1=32, hidden2=64, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K = K; self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool1d(K)
        self.fc    = nn.Linear(hidden2 * K, out_dim)

    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1)
        h = self.pool(h).flatten(1)
        return F.normalize(self.fc(h), p=2, dim=1)


class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(
            nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
            nn.Linear(hidden_mlp, n_bins))
    def forward(self, a, b):
        return self.head(torch.abs(self.encoder(a) - self.encoder(b)))
    def encode(self, x):
        return self.encoder(x)


class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]
        self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]


def train_encoder(alphabet, band_low, label, train_seed, n_epochs=NUM_EPOCHS):
    torch.manual_seed(train_seed)
    rng = np.random.default_rng(train_seed)
    print(f'\n===== Training {label} encoder (alphabet={alphabet}, band_low={band_low}, seed={train_seed}) =====')
    pairs = make_full_range_pairs(alphabet, N_TRAIN, rng)
    labels = np.array([p[2] for p in pairs])
    counts = {b: int(c) for b, c in zip(BIN_NAMES,
              np.bincount([bin_idx_for(l, band_low) for l in labels], minlength=3))}
    print(f'  {len(pairs)} pairs, normLev in [{labels.min():.3f}, {labels.max():.3f}], bins={counts}')
    dl = DataLoader(PairDatasetCE(pairs, band_low), batch_size=BATCH_SIZE, shuffle=True)
    model = SiameseClassifier(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(1, n_epochs + 1):
        tot = 0.0; nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{label}] epoch {epoch:3d}/{n_epochs} CE={tot/nb:.4f}')
    model.eval()
    return model

## 5. Data load — pool, eval set, 3Di per-pair score (from colab19)

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
prot_df = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')
prot_df = prot_df[prot_df['aa_seq'].apply(is_standard_aa)]
prot_df = prot_df[prot_df['ss_seq'].apply(is_standard_ss)]
prot_df = prot_df[prot_df['aa_seq'].str.len() == prot_df['ss_seq'].str.len()]
prot_df['aa_len'] = prot_df['aa_seq'].str.len()
RESCUED = {'4z0mC02', '3qkaE02'}
in_range = prot_df['aa_len'].between(MIN_LEN, MAX_LEN)
prot_df = prot_df[in_range | prot_df['domain_id'].isin(RESCUED)].reset_index(drop=True)

id_to_aa = dict(zip(prot_df['domain_id'], prot_df['aa_seq']))
id_to_ss = dict(zip(prot_df['domain_id'], prot_df['ss_seq']))
all_domains = list(prot_df['domain_id'])
print(f'Protein pool: {len(prot_df)}')

eval_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
seqs3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_3di.csv.gz'))
id_to_3di = dict(zip(seqs3['domain_id'], seqs3['3di'].astype(str)))

def pair_3di_score(a, b):
    if a in id_to_3di and b in id_to_3di:
        return norm_lev(id_to_3di[a], id_to_3di[b])
    return np.nan
eval_df['3di_score'] = [pair_3di_score(a, b) for a, b in zip(eval_df['domain_a'], eval_df['domain_b'])]

SCORE_COL = {'AA': 'aa_score', 'SS': 'ss_score', '3Di': '3di_score'}
LOOKUP    = {'AA': id_to_aa,  'SS': id_to_ss,  '3Di': id_to_3di}
for feed, sc in SCORE_COL.items():
    n = int((eval_df[sc] >= BAND_HIGH).sum())
    tag = 'powered' if n >= 200 else ('moderate' if n >= 30 else 'underpowered (n<30)')
    print(f'  {feed}-feed: {n} eval pairs >= {BAND_HIGH}  [{tag}]')

## 6. Train the two encoders (frozen afterwards)

In [ ]:
model_aa = train_encoder(AA_ALPHABET, BAND_LOW_AA, 'AA', SEED_TRAIN_AA)
model_ss = train_encoder(SS_ALPHABET, BAND_LOW_SS, 'SS', SEED_TRAIN_SS)

## 7. Metric machinery — predicted L2-sim, AUROC, eval-pair selection

- `pred_sim_strings`: deployment score `1 - ||e_a - e_b|| / 2` for arbitrary pairs.
- `encode_pool`: batch-encode the full retrieval pool once per (encoder, feed).
- `auroc_cell`: **Q1** number. is-high AUROC over the full eval set of a feed
  (positives = `>= 0.70`, negatives = the rest), discriminator = predicted L2-sim.
- `feed_high_pairs`: the `>= 0.70` eval pairs with both proteins in pool (the retrieval queries).

In [ ]:
@torch.no_grad()
def encode_pool(model, seq_lookup, ids, batch_size=256):
    model.eval(); valid = [d for d in ids if d in seq_lookup]; out = []
    for i in range(0, len(valid), batch_size):
        x = torch.stack([encode_pad(seq_lookup[d]) for d in valid[i:i+batch_size]]).to(device)
        out.append(model.encoder(x))
    return (torch.cat(out, 0) if out else torch.empty(0)), valid

@torch.no_grad()
def pred_sim_strings(model, A, B, batch=512):
    model.eval(); sims = []
    for i in range(0, len(A), batch):
        xa = torch.stack([encode_pad(s) for s in A[i:i+batch]]).to(device)
        xb = torch.stack([encode_pad(s) for s in B[i:i+batch]]).to(device)
        d = torch.linalg.vector_norm(model.encoder(xa) - model.encoder(xb), dim=1)
        sims.append((1.0 - d / 2.0).cpu().numpy())
    return np.concatenate(sims) if sims else np.array([])

def feed_eval_pairs(feed):
    """All eval pairs scored in this alphabet with both proteins available."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    sub = eval_df.dropna(subset=[sc])
    return sub[sub['domain_a'].isin(lk) & sub['domain_b'].isin(lk)]

def feed_high_pairs(feed):
    """>= 0.70 eval pairs with both proteins in the retrieval pool (the queries)."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    inpool = set(all_domains) & set(lk)
    sub = eval_df.dropna(subset=[sc])
    return sub[(sub[sc] >= BAND_HIGH) & sub['domain_a'].isin(inpool) & sub['domain_b'].isin(inpool)]

def auroc_cell(model, feed):
    """Q1 - is-high AUROC. Returns (auroc, n_pos, n_total)."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    sub = feed_eval_pairs(feed)
    y = (sub[sc].values >= BAND_HIGH).astype(int)
    if y.sum() == 0 or y.sum() == len(y): return np.nan, int(y.sum()), len(y)
    pred = pred_sim_strings(model, [lk[d] for d in sub['domain_a']], [lk[d] for d in sub['domain_b']])
    return float(roc_auc_score(y, pred)), int(y.sum()), len(y)

## 8. Retrieval — exact-Levenshtein oracle neighbour sets (uniform, no AA special-casing)

For each query we brute-force `normLev` of the query against all ~10k pool proteins and take
**everything `>= 0.70`** as the true neighbour set `T`. Same definition for AA, SS, 3Di — AA just
ends up with `|T|~1` because high-AA pairs are rare. `median |T|` is the crowding of each feed.

In [ ]:
def build_oracle_cache(feed):
    """Exact-Levenshtein true neighbour sets (>= BAND_HIGH) per query, over the full pool."""
    lk = LOOKUP[feed]
    pool_ids = [d for d in all_domains if d in lk]
    idx = {d: i for i, d in enumerate(pool_ids)}
    pool_seqs = [lk[d] for d in pool_ids]
    lens = np.array([len(s) for s in pool_seqs])
    sub = feed_high_pairs(feed)
    q_ids = sorted({d for pr in sub[['domain_a', 'domain_b']].values for d in pr})
    if not q_ids:
        return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': [], 'true_sets': {}}
    D = rf_cdist([lk[q] for q in q_ids], pool_seqs,
                 scorer=RFLevenshtein.distance, workers=-1).astype(float)
    true_sets = {}
    for r, q in enumerate(q_ids):
        qi = idx[q]
        denom = np.maximum(lens, lens[qi]); denom[denom == 0] = 1
        osim = 1.0 - D[r] / denom; osim[qi] = -np.inf
        true_sets[q] = set(np.where(osim >= BAND_HIGH)[0].tolist())
    nt = [len(true_sets[q]) for q in q_ids]
    print(f'  oracle[{feed:<3}]: {len(q_ids):>3} queries, median |T|={int(np.median(nt))}, max={max(nt)}')
    return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': q_ids, 'true_sets': true_sets}

print('Building exact-Levenshtein oracle neighbour sets (all feeds, uniform)...')
ORACLE = {feed: build_oracle_cache(feed) for feed in ['AA', 'SS', '3Di']}

## 9. Retrieval metrics — MAP@10, R-precision, recall@k, precision@k, hit-rate@k

All scored against the oracle set, all rank from the encoder's L2 geometry (`1 - ||e_a-e_b||/2`).

- **MAP@k** (headline): rank-aware. `AP@k = (1/min(|T|,k)) * sum_i prec@i * rel(i)`, averaged
  over queries. Rewards ranking genuine neighbours *first*, not just being in the top-k bag.
- **R-precision**: give the encoder exactly `|T|` slots, count how many are genuine -> the
  crowding-fair "of N partners, found M" number (no fixed-k cap).
- **recall@k** = found / `|T|` (capped at `k/|T|` when crowded); **precision@k** = found / k;
  **hit-rate@k** = found *any* (the lenient floor).

In [ ]:
K_LIST = (10, 50)     # report depths for recall/precision/hit-rate
K_MAP  = 10           # headline MAP@k depth

def _ap_at_k(order, true_set, k):
    """Average Precision @k for one query. order = ranked pool indices (self excluded)."""
    nt = len(true_set)
    if nt == 0:
        return np.nan
    hits = 0; ap = 0.0
    for i, oi in enumerate(order[:k], start=1):
        if oi in true_set:
            hits += 1
            ap += hits / i
    return ap / min(nt, k)

def retrieval_metrics(model, feed, cache, k_list=K_LIST, k_map=K_MAP):
    """Set-based retrieval vs the exact-Lev oracle, uniform across feeds."""
    lk = LOOKUP[feed]
    if not cache['q_ids']:
        return None
    pool_emb, _ = encode_pool(model, lk, cache['pool_ids'])
    idx = cache['idx']
    nts, ap_map, rprec = [], [], []
    recall    = {k: [] for k in k_list}
    precision = {k: [] for k in k_list}
    hit       = {k: [] for k in k_list}
    for q in cache['q_ids']:
        qi = idx[q]; ts = cache['true_sets'][q]; nt = len(ts)
        if nt == 0:
            continue
        nts.append(nt)
        esim = (1.0 - torch.linalg.vector_norm(pool_emb - pool_emb[qi], dim=1) / 2.0).cpu().numpy()
        esim[qi] = -np.inf
        order = np.argsort(-esim)
        ap_map.append(_ap_at_k(order, ts, k_map))
        topT = set(order[:nt].tolist())
        rprec.append(len(topT & ts) / nt)                 # R-precision
        for k in k_list:
            ret = len(set(order[:k].tolist()) & ts)
            recall[k].append(ret / nt)
            precision[k].append(ret / k)
            hit[k].append(1.0 if ret >= 1 else 0.0)
    out = {'n_q': len(nts), 'median_n_true': float(np.median(nts)),
           f'MAP@{k_map}': float(np.mean(ap_map)), 'Rprec': float(np.mean(rprec))}
    for k in k_list:
        out[f'recall@{k}']    = float(np.mean(recall[k]))
        out[f'precision@{k}'] = float(np.mean(precision[k]))
        out[f'hitrate@{k}']   = float(np.mean(hit[k]))
    return out

## 10. Compute the 2x3 matrix — Accuracy (AUROC) + Retrieval (oracle-set)

In [ ]:
ENCODERS = [('AA-enc', model_aa), ('SS-enc', model_ss)]
FEEDS = ['AA', 'SS', '3Di']

results = []
for enc_label, model in ENCODERS:
    for feed in FEEDS:
        auroc, n_pos, n_tot = auroc_cell(model, feed)
        rm = retrieval_metrics(model, feed, ORACLE[feed]) or {}
        in_domain = (enc_label, feed) in [('AA-enc', 'AA'), ('SS-enc', 'SS')]
        row = {'encoder': enc_label, 'feed': feed,
               'role': 'in-domain' if in_domain else 'cross-rep',
               'auroc': auroc, 'auroc_n_pos': n_pos, 'auroc_n_total': n_tot}
        row.update(rm)
        results.append(row)
        print(f"{enc_label} x {feed:<4} | AUROC={auroc:.3f} (n_pos={n_pos}) | "
              f"MAP@{K_MAP}={rm['MAP@%d' % K_MAP]:.3f}  R-prec={rm['Rprec']:.3f}  "
              f"rec@10={rm['recall@10']:.3f} prec@10={rm['precision@10']:.3f} "
              f"hit@10={rm['hitrate@10']:.3f} | med|T|={rm['median_n_true']:.0f} (n_q={rm['n_q']})")

res_df = pd.DataFrame(results)

## 11. Chart 1 — Accuracy (AUROC is-high)

Unchanged from colab22. Grouped bars: blue = AA-encoder, green = SS-encoder, dashed = chance.
**AA bars are n_pos=5** (only 5 high-AA pairs exist) -> read AA as underpowered; SS (n=333) and
3Di (n=38) are the trustworthy columns.

In [ ]:
def grouped_vals(metric):
    return ([next(r[metric] for r in results if r['encoder']=='AA-enc' and r['feed']==f) for f in FEEDS],
            [next(r[metric] for r in results if r['encoder']=='SS-enc' and r['feed']==f) for f in FEEDS])

aa_auroc, ss_auroc = grouped_vals('auroc')
x = np.arange(len(FEEDS)); w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - w/2, aa_auroc, w, label='AA-encoder', color=COL_AA_ENC)
b2 = ax.bar(x + w/2, ss_auroc, w, label='SS-encoder', color=COL_SS_ENC)
ax.axhline(0.5, ls='--', color='grey', lw=1)
ax.text(len(FEEDS)-0.5, 0.515, 'chance (0.5)', color='grey', fontsize=9, ha='right')
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=9)
ax.set_ylim(0, 1.05); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel('AUROC  (is-high >= 0.70)'); ax.set_xlabel('Feed modality')
ax.set_xticks(x); ax.set_xticklabels(FEEDS)
ax.set_title('Accuracy: can embedding distance separate a high-similarity pair from a random one?')
for i, f in enumerate(FEEDS):
    npos = next(r['auroc_n_pos'] for r in results if r['feed']==f)
    ax.annotate(f'n_pos={npos}', (i, 0), (0, -28), textcoords='offset points',
                ha='center', va='top', fontsize=8, color='dimgray')
ax.legend(loc='lower left', framealpha=0.9)
plt.tight_layout(); plt.savefig('colab23_accuracy_auroc.png', dpi=150, bbox_inches='tight'); plt.show()

## 12. Chart 2 — Retrieval quality (MAP@10)

The headline retrieval number, **one rank-aware metric across all three feeds** (replaces the
colab22 hits@10/frac@10 split). `med|T|` under each group = crowding (median genuine neighbours
per query). A perfect Levenshtein-approximator scores 1.0, so a bar reads directly as
"fraction of ideal Lev-ranking achieved" — no separate oracle ceiling needed.

In [ ]:
map_key = f'MAP@{K_MAP}'
aa_map, ss_map = grouped_vals(map_key)
aa_map = [100*v for v in aa_map]; ss_map = [100*v for v in ss_map]
x = np.arange(len(FEEDS)); w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - w/2, aa_map, w, label='AA-encoder', color=COL_AA_ENC)
b2 = ax.bar(x + w/2, ss_map, w, label='SS-encoder', color=COL_SS_ENC)
ax.bar_label(b1, fmt='%.0f%%', padding=2, fontsize=9)
ax.bar_label(b2, fmt='%.0f%%', padding=2, fontsize=9)
ax.set_ylim(0, 110); ax.set_yticks(np.arange(0, 101, 20))
ax.set_ylabel(f'{map_key}  (rank-aware, vs exact-Lev neighbour set)')
ax.set_xlabel('Feed modality')
ax.set_xticks(x); ax.set_xticklabels(FEEDS)
ax.set_title('Retrieval quality: does embedding distance rank the true Levenshtein neighbours first?')
for i, f in enumerate(FEEDS):
    mt = next(r['median_n_true'] for r in results if r['feed']==f)
    ax.annotate(f'med|T|={mt:.0f}', (i, 0), (0, -28), textcoords='offset points',
                ha='center', va='top', fontsize=8, color='dimgray')
ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout(); plt.savefig('colab23_retrieval_map.png', dpi=150, bbox_inches='tight'); plt.show()

## 13. Summary table + CSV export — full retrieval suite

In [ ]:
print('=' * 124)
print('COLAB23 SUMMARY — 2x3 matrix; retrieval vs exact-Lev oracle neighbour set (uniform, no AA special-casing)')
print('=' * 124)
print(f'{"enc":<8}{"feed":<6}{"role":<11}{"AUROC":>7}{"n_pos":>6}'
      f'{"MAP@10":>8}{"R-prec":>8}{"rec@10":>8}{"rec@50":>8}{"prec@10":>9}{"hit@10":>8}{"med|T|":>8}{"n_q":>6}')
print('-' * 124)
for r in results:
    print(f"{r['encoder']:<8}{r['feed']:<6}{r['role']:<11}{r['auroc']:>7.3f}{r['auroc_n_pos']:>6}"
          f"{r['MAP@10']:>8.3f}{r['Rprec']:>8.3f}{r['recall@10']:>8.3f}{r['recall@50']:>8.3f}"
          f"{r['precision@10']:>9.3f}{r['hitrate@10']:>8.3f}{r['median_n_true']:>8.0f}{r['n_q']:>6}")

res_df.to_csv('colab23_summary.csv', index=False)
print('\nSaved: colab23_summary.csv, colab23_accuracy_auroc.png, colab23_retrieval_map.png')

## 14. Embedding-space analysis — UMAP + what the axes mean

Revives the colab16 pool-UMAP (length-coloured, the high-AA partner pairs drawn as red segments)
and reads the space more carefully:

1. **Figure** — the map itself.
2. **Axis meaning** — correlate each UMAP axis with length and with composition; which axis is which?
3. **Horizontal vs vertical** — does moving along one axis change *length* while the other changes
   *character content*? (Directly tests the "same length but different characters" question.)
4. **Neighbourhood** — for a high-AA query, what are the *other* sequences sitting nearby? (Length
   twins, or genuine Lev-neighbours?)
5. **Local zoom** — one pair's neighbourhood, coloured by exact normLev-to-query, so the crowd is visible.

All on the **AA encoder, AA feed** (the well-posed case where the red-segment story is cleanest).

In [ ]:
import umap
from sklearn.decomposition import PCA
from scipy.stats import spearmanr

# ---- composition helpers (alphabet-agnostic content descriptors) ----
def comp_hist(seq):
    v = np.zeros(len(AA_ALPHABET))
    for c in seq:
        j = CHAR_TO_IDX.get(c, -1)
        if j >= 0: v[j] += 1
    s = v.sum()
    return v / s if s > 0 else v

def shannon_entropy(seq):
    h = comp_hist(seq); h = h[h > 0]
    return float(-(h * np.log2(h)).sum()) if h.size else 0.0

# ---- substrate: AA-encoder pool embedding + 2D UMAP + descriptors ----
VIZ_MODEL = model_aa
pool_ids_viz = [d for d in all_domains if d in id_to_aa]
idx_viz = {d: i for i, d in enumerate(pool_ids_viz)}
pool_emb_viz, _ = encode_pool(VIZ_MODEL, id_to_aa, pool_ids_viz)
pool_emb_viz = pool_emb_viz.cpu().numpy()
print(f'pool embeddings: {pool_emb_viz.shape}')

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED)
pool_2d = reducer.fit_transform(pool_emb_viz)
id_to_xy = {d: pool_2d[i] for i, d in enumerate(pool_ids_viz)}

lens_viz = np.array([len(id_to_aa[d]) for d in pool_ids_viz])
comp_mat = np.stack([comp_hist(id_to_aa[d]) for d in pool_ids_viz])          # (N, 20)
ent_viz  = np.array([shannon_entropy(id_to_aa[d]) for d in pool_ids_viz])
comp_pc  = PCA(n_components=2, random_state=SEED).fit_transform(comp_mat)     # composition axes

HIGH_PAIRS_AA = [tuple(p) for p in feed_high_pairs('AA')[['domain_a', 'domain_b']].values]
print(f'high-AA pairs for overlay: {len(HIGH_PAIRS_AA)}')

### 14.1 Figure — pool UMAP, high-AA pairs as red segments (the colab16 revival)

In [ ]:
ux, uy = pool_2d[:, 0], pool_2d[:, 1]
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(ux, uy, c=lens_viz, cmap='viridis', s=8, alpha=0.5, linewidths=0)
plt.colorbar(sc, ax=ax, label='Protein length (AA)')
for a, b in HIGH_PAIRS_AA:
    if a in id_to_xy and b in id_to_xy:
        xa, ya = id_to_xy[a]; xb, yb = id_to_xy[b]
        ax.plot([xa, xb], [ya, yb], color='red', lw=1.5, alpha=0.9, zorder=5)
        ax.scatter([xa, xb], [ya, yb], color='red', s=80, edgecolors='black', linewidths=1.2, zorder=6)
        ax.annotate(f'{a[:6]}\u2194{b[:6]}', ((xa+xb)/2, (ya+yb)/2), fontsize=8, ha='center',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='red', alpha=0.8))
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_title('AA-encoder pool embedding (UMAP) — high-AA partner pairs as red segments')
plt.tight_layout(); plt.savefig('colab23_umap_pool.png', dpi=150, bbox_inches='tight'); plt.show()

### 14.2 Axis meaning — which UMAP axis is length, which is composition?

Spearman of each axis with length, with the two composition principal components, and with
sequence entropy. (UMAP orientation is arbitrary per run, so we *measure* it rather than assume.)

In [ ]:
def rho(a, b): return float(spearmanr(a, b).correlation)

print('Spearman correlation of UMAP axes with sequence descriptors')
print(f'{"":<14}{"UMAP-1":>9}{"UMAP-2":>9}')
for name, vec in [('length', lens_viz), ('comp-PC1', comp_pc[:, 0]),
                  ('comp-PC2', comp_pc[:, 1]), ('entropy', ent_viz)]:
    print(f'{name:<14}{rho(ux, vec):>9.3f}{rho(uy, vec):>9.3f}')

len_axis  = 'UMAP-1' if abs(rho(ux, lens_viz)) >= abs(rho(uy, lens_viz)) else 'UMAP-2'
comp_axis = 'UMAP-2' if len_axis == 'UMAP-1' else 'UMAP-1'
print(f'\n-> LENGTH is carried mainly by {len_axis}; {comp_axis} is the orthogonal '
      f'(composition/content) axis.')

### 14.3 Horizontal vs vertical — what does moving along each axis change?

Sample many random protein pairs; for each, measure the UMAP displacement (|dx|, |dy|), the length
gap, and the composition distance (cosine on AA-frequency histograms). Compare the most-horizontal
displacements against the most-vertical ones, then — holding length fixed — check which axis still
tracks composition. This is the direct answer to "is a horizontal move same-length-but-different-
characters, or is it a length change?"

In [ ]:
rng_viz = np.random.default_rng(SEED)
N_PAIRS = 30000
ii = rng_viz.integers(0, len(pool_ids_viz), N_PAIRS)
jj = rng_viz.integers(0, len(pool_ids_viz), N_PAIRS)
ok = ii != jj; ii, jj = ii[ok], jj[ok]

dx   = np.abs(ux[ii] - ux[jj])
dy   = np.abs(uy[ii] - uy[jj])
dlen = np.abs(lens_viz[ii] - lens_viz[jj]).astype(float)

# composition distance, vectorised: 1 - cosine on the precomputed histograms
ca, cb = comp_mat[ii], comp_mat[jj]
na = np.linalg.norm(ca, axis=1); nb = np.linalg.norm(cb, axis=1)
denom = np.clip(na * nb, 1e-9, None)
dcomp = 1.0 - (ca * cb).sum(1) / denom

ratio = (dx + 1e-9) / (dy + 1e-9)
horiz = ratio > np.quantile(ratio, 0.8)   # top 20% most-horizontal displacements
vert  = ratio < np.quantile(ratio, 0.2)   # top 20% most-vertical

print('Displacement-direction probe (what changes when you move along each axis)')
print(f'{"group":<22}{"mean d-length":>14}{"mean d-comp":>13}')
print(f'{"mostly-horizontal":<22}{dlen[horiz].mean():>14.1f}{dcomp[horiz].mean():>13.3f}')
print(f'{"mostly-vertical":<22}{dlen[vert].mean():>14.1f}{dcomp[vert].mean():>13.3f}')
print(f'{"all pairs":<22}{dlen.mean():>14.1f}{dcomp.mean():>13.3f}')

same_len = dlen <= 3
print(f'\nAmong near-same-length pairs (|d-length| <= 3, n={int(same_len.sum())}):')
print(f'  corr(|dx|, d-comp) = {np.corrcoef(dx[same_len], dcomp[same_len])[0, 1]:+.3f}')
print(f'  corr(|dy|, d-comp) = {np.corrcoef(dy[same_len], dcomp[same_len])[0, 1]:+.3f}')
print('  -> at fixed length, the axis whose displacement tracks d-comp is the content axis;')
print('     i.e. two same-length sequences with totally different characters separate along THAT axis.')

### 14.4 Neighbourhood — what are the *other* sequences near a high-AA pair?

For each high-AA query, list its top embedding neighbours with: embedding-sim, length, **exact
normLev to the query**, and composition-sim. If neighbours are length-twins with *low* normLev
(and only the true partner is high), that is the crowding the retrieval metric has to fight.

In [ ]:
N_NB = 12
def neighbours_table(query, partner):
    qi = idx_viz[query]
    esim = 1.0 - np.linalg.norm(pool_emb_viz - pool_emb_viz[qi], axis=1) / 2.0
    esim[qi] = -np.inf
    order = np.argsort(-esim)[:N_NB]
    qpart_nl = norm_lev(id_to_aa[query], id_to_aa[partner])
    print(f'\nQuery {query} (len {len(id_to_aa[query])}) | partner {partner} '
          f'(exact normLev {qpart_nl:.3f})')
    print(f'  {"rank":<5}{"domain":<10}{"emb-sim":>8}{"len":>5}{"normLev":>9}{"comp-sim":>9}  note')
    for rk, oi in enumerate(order, 1):
        d = pool_ids_viz[oi]
        nl = norm_lev(id_to_aa[query], id_to_aa[d])
        cs = float((comp_mat[qi] * comp_mat[oi]).sum() /
                   max(1e-9, np.linalg.norm(comp_mat[qi]) * np.linalg.norm(comp_mat[oi])))
        note = '<== TRUE PARTNER' if d == partner else ('high-sim' if nl >= BAND_HIGH else '')
        print(f'  {rk:<5}{d:<10}{esim[oi]:>8.3f}{len(id_to_aa[d]):>5}{nl:>9.3f}{cs:>9.3f}  {note}')

for a, b in HIGH_PAIRS_AA[:3]:
    neighbours_table(a, b)

### 14.5 Local zoom — one pair's neighbourhood, coloured by exact normLev-to-query

Zooms into the first high-AA pair. Points coloured by exact normLev to the query: the true partner
(red) is high, but most surrounding points are length-twins with low normLev — the visible "crowd"
that makes single-target retrieval hard even when the partner is correctly placed nearby.

In [ ]:
qa, qb = HIGH_PAIRS_AA[0]
cx, cy = id_to_xy[qa]
span_x = (ux.max() - ux.min()) * 0.12
span_y = (uy.max() - uy.min()) * 0.12
near = (np.abs(ux - cx) < span_x) & (np.abs(uy - cy) < span_y)
near_idx = np.where(near)[0]
nl_to_q = np.array([norm_lev(id_to_aa[qa], id_to_aa[pool_ids_viz[i]]) for i in near_idx])

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(ux[near_idx], uy[near_idx], c=nl_to_q, cmap='magma', vmin=0, vmax=1,
                s=30, alpha=0.85, linewidths=0)
plt.colorbar(sc, ax=ax, label=f'exact normLev to query {qa}')
xa, ya = id_to_xy[qa]; xb, yb = id_to_xy[qb]
ax.plot([xa, xb], [ya, yb], color='red', lw=2, zorder=5)
ax.scatter([xa, xb], [ya, yb], color='red', s=130, edgecolors='black', linewidths=1.3, zorder=6)
ax.annotate('query', (xa, ya), fontsize=9, color='red')
ax.annotate('partner', (xb, yb), fontsize=9, color='red')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_title(f'Local neighbourhood of {qa} <-> {qb}\n(most nearby points are length-twins with low normLev)')
plt.tight_layout(); plt.savefig('colab23_umap_zoom.png', dpi=150, bbox_inches='tight'); plt.show()
print(f'local window: {len(near_idx)} points; '
      f'{int((nl_to_q >= BAND_HIGH).sum())} of them are genuine high-sim (>= {BAND_HIGH})')

## How to read this notebook

**Retrieval (Sections 8-13).** One rank-aware metric family, scored against the exact-Levenshtein
oracle neighbour set, uniform across AA / SS / 3Di:

- **MAP@10** is the headline (rewards ranking genuine neighbours first; the supervisor's metric).
- **R-precision** is the intuitive "of N genuine partners, how many in the top N" — crowding-fair.
- **recall@k / precision@k** are the decomposed companions; **hit-rate@10** is the lenient floor
  (found *any*). `med|T|` reports crowding so cross-feed bars are read in context.
- Because relevance is defined by the same metric the encoder approximates, a perfect approximator
  scores 1.0 — there is no separate "oracle ceiling" row to chase (that was the §6 designated-partner
  framing; this is the metric-preservation framing, which is the actual thesis claim).

**Space (Section 14).** Length is the dominant axis of the embedding (high-sim requires near-equal
length, so partners share a length band). The added analysis separates *length* from *content*:
the axis-correlation table says which UMAP axis is which; the displacement probe answers whether a
horizontal move is a length change or a same-length content change; the neighbourhood table and the
zoom show that a true partner sits inside a *crowd of length-twins with low normLev* — which is
exactly why SS/3Di retrieval is hard (many length-twins) while AA is easy (the partner is the only
high-normLev neighbour in its length band).